# Target Engineering Quality: $1T Smoke EDA

This gated warehouse notebook measures target construction cost, class support, yearly stability, symbol coverage, and overlap. It does not train a model or select targets using in-sample classifier performance.

In [1]:
from __future__ import annotations
from pathlib import Path
from time import perf_counter
import os
import sys
import numpy as np
import pandas as pd
from IPython.display import display

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / 'pyproject.toml').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from quant_warehouse.platforms.data_providers.fmp.target_engineering.event_pairs import EventPairStore
from quant_warehouse.research_tools import (
    BinaryTargetConfig, FamilyEvaluationConfig, build_event_target_panel,
    build_oracle_trade_target_panel, combine_target_panels, load_fmp_event_pairs,
    screen_fmp_equity_universe, summarize_binary_targets,
)
from quant_warehouse.warehouse.api import Warehouse

MIN_MARKET_CAP = 1_000_000_000_000
START_DATE = '2018-01-01'
END_DATE = None
MIN_POSITIVES_PER_YEAR = 5

def rss_gib() -> float:
    value = next(line for line in Path('/proc/self/status').read_text().splitlines() if line.startswith('VmRSS:')).split()[1]
    return float(value) / 1024 / 1024

print({'python': sys.version.split()[0], 'architecture': os.uname().machine, 'rss_gib': round(rss_gib(), 3)})

{'python': '3.12.13', 'architecture': 'aarch64', 'rss_gib': 0.307}


In [2]:
warehouse = Warehouse()
feature_config = FamilyEvaluationConfig(market_cap_min=MIN_MARKET_CAP, start_date=START_DATE, end_date=END_DATE)
symbols, raw_universe, eligibility, universe_source = screen_fmp_equity_universe(feature_config, warehouse=warehouse, required_sections=('prices',))
base_parts = []
for symbol in symbols:
    prices = warehouse.read_prices(symbol, provider='fmp', start=START_DATE, end=END_DATE)
    if prices is None or prices.empty:
        continue
    base_parts.append(pd.DataFrame({'symbol': symbol, 'date': pd.to_datetime(prices.index).normalize()}))
feature_dates = pd.concat(base_parts, ignore_index=True).drop_duplicates(['symbol', 'date'])
print({'universe_source': universe_source, 'symbols': len(symbols), 'symbol_days': len(feature_dates)})

{'universe_source': 'openbb:fmp', 'symbols': 14, 'symbol_days': 27855}


## Build event-pair and oracle-entry targets

In [3]:
target_config = BinaryTargetConfig(
    provider='fmp', start_date=START_DATE, end_date=END_DATE,
    event_families=('congress', 'insider', 'analyst_rating', 'price_target', 'earnings'),
    oracle_trade_k_by_frequency={'YE': (1, 3, 6, 12)},
)
peak_rss = rss_gib()
started = perf_counter()
events, event_diagnostics, event_seconds = load_fmp_event_pairs(
    symbols, target_config, event_store=EventPairStore(backend=warehouse.backend, catalog=warehouse.catalog), include_historical=True
)
event_panel, event_metadata = build_event_target_panel(feature_dates, events, target_config)
peak_rss = max(peak_rss, rss_gib())
oracle_panel, oracle_metadata, oracle_seconds = build_oracle_trade_target_panel(symbols, target_config, warehouse=warehouse)
peak_rss = max(peak_rss, rss_gib())
targets = combine_target_panels(event_panel, oracle_panel)
metadata = pd.concat([event_metadata, oracle_metadata], ignore_index=True).drop_duplicates('target')
oracle_targets = oracle_metadata['target'].tolist()
assert oracle_targets == ['target_oracle_trade_entry__long_vs_short']
assert not any('_any' in target or '_k' in target for target in oracle_targets)
summary = summarize_binary_targets(targets, metadata)
event_coverage = (
    events.groupby(['event_family', 'event_type'], dropna=False)
    .agg(rows=('event_type', 'size'), symbols=('symbol', 'nunique'), min_date=('event_date', 'min'), max_date=('event_date', 'max'))
    .reset_index()
)
print({'events': len(events), 'targets': len(metadata), 'event_seconds': round(event_seconds, 3), 'oracle_seconds': round(oracle_seconds, 3), 'total_seconds': round(perf_counter()-started, 3), 'peak_rss_gib': round(peak_rss, 3)})
display(event_diagnostics)
display(event_coverage)
display(summary)

{'events': 5649, 'targets': 11, 'event_seconds': 1.02, 'oracle_seconds': 1.271, 'total_seconds': 2.391, 'peak_rss_gib': 0.681}


,symbol,cached_rows,historical_rows,combined_rows,event_families
0,AAPL,32,907,939,"(analyst_rating, congress, earnings, insider, ..."
1,AMZN,34,756,790,"(analyst_rating, congress, earnings, insider, ..."
2,AVGO,0,1,1,"(earnings,)"
3,BRK-A,0,1,1,"(earnings,)"
4,BRK-B,0,17,17,"(congress,)"
5,GOOG,0,0,0,()
6,GOOGL,34,622,656,"(analyst_rating, congress, earnings, insider, ..."
7,LLY,0,166,166,"(congress,)"
8,META,34,450,484,"(analyst_rating, congress, earnings, insider, ..."
9,MSFT,34,1078,1112,"(analyst_rating, congress, earnings, insider, ..."


,event_family,event_type,rows,symbols,min_date,max_date
0,analyst_rating,analyst_downgrade,117,7,2024-04-03,2026-06-25
1,analyst_rating,analyst_upgrade,401,7,2024-04-04,2026-06-09
2,congress,congress_buy,1707,9,2018-01-25,2026-06-10
3,congress,congress_sell,1676,9,2018-01-04,2026-06-16
4,earnings,earnings_beat,191,9,2018-01-31,2026-05-20
5,earnings,earnings_miss,36,7,2018-02-01,2026-02-05
6,insider,insider_buy,217,7,2018-09-30,2026-06-16
7,insider,insider_sell,859,7,2018-08-02,2026-06-18
8,price_target,price_target_cut,95,7,2024-01-02,2026-06-25
9,price_target,price_target_raise,350,7,2024-01-11,2026-06-22


,target,target_family,rows,rate_rows,positive_rows,positive_rate,positive_symbols,min_positive_date,max_positive_date
0,target_event_on__congress_buy,event,27855,2664,1458,0.547297,9,2018-01-25,2026-06-10
1,target_event_on__congress_sell,event,27855,2664,1431,0.537162,9,2018-01-04,2026-06-16
2,target_event_on__insider_sell,event,27855,905,736,0.813260,7,2018-08-02,2026-06-18
3,target_event_on__analyst_upgrade,event,27855,269,214,0.795539,7,2024-04-04,2026-06-09
4,target_event_on__price_target_raise,event,27855,246,202,0.821138,7,2024-01-11,2026-06-22
5,target_event_on__earnings_beat,event,27855,227,191,0.841410,9,2018-01-31,2026-05-20
6,target_event_on__insider_buy,event,27855,905,185,0.204420,7,2018-10-01,2026-06-16
7,target_event_on__analyst_downgrade,event,27855,269,69,0.256506,7,2024-04-03,2026-06-25
8,target_event_on__price_target_cut,event,27855,246,61,0.247967,7,2024-01-02,2026-06-25
9,target_event_on__earnings_miss,event,27855,227,36,0.158590,7,2018-02-01,2026-02-05


## Annual support, stability, and target overlap

In [4]:
target_columns = [column for column in metadata['target'] if column in targets.columns]
work = targets[['symbol', 'date', *target_columns]].copy()
work['year'] = pd.to_datetime(work['date']).dt.year
annual = work.groupby('year')[target_columns].sum().T.reset_index(names='target')
year_columns = [column for column in annual.columns if isinstance(column, (int, np.integer))]
annual['years_with_support'] = annual[year_columns].ge(MIN_POSITIVES_PER_YEAR).sum(axis=1)
annual['total_positives'] = annual[year_columns].sum(axis=1)
annual = annual.merge(metadata, on='target', how='left').sort_values(['years_with_support', 'total_positives'], ascending=False)
display(annual)

binary = work[target_columns].fillna(0).gt(0).astype('uint8')
active = binary.sum(axis=0)
overlap_rows = []
for left_index, left in enumerate(target_columns):
    for right in target_columns[left_index + 1:]:
        intersection = int((binary[left] & binary[right]).sum())
        union = int((binary[left] | binary[right]).sum())
        if intersection:
            overlap_rows.append({'left': left, 'right': right, 'intersection': intersection, 'jaccard': intersection / union if union else np.nan})
overlap = pd.DataFrame(overlap_rows).sort_values(['jaccard', 'intersection'], ascending=False) if overlap_rows else pd.DataFrame()
display(overlap.head(50))

gate = annual[['target', 'target_family', 'years_with_support', 'total_positives']].copy()
gate['cheap_status'] = np.select(
    [gate['total_positives'].lt(20), gate['years_with_support'].lt(3)],
    ['insufficient_total_support', 'unstable_annual_support'],
    default='candidate_for_wfo',
)
display(gate.groupby(['target_family', 'cheap_status']).size().unstack(fill_value=0))
display(gate.sort_values(['cheap_status', 'total_positives']).head(80))

,target,2018,2019,2020,2021,2022,2023,2024,2025,2026,years_with_support,total_positives,target_family,target_type
0,target_event_on__congress_buy,75,57,196,114,267,202,155,322,70,9,1458,event,binary
1,target_event_on__congress_sell,39,75,185,155,250,219,190,265,53,9,1431,event,binary
10,target_oracle_trade_entry__long_vs_short,156,156,156,155,156,153,152,154,112,9,1350,oracle_trade,binary
3,target_event_on__insider_sell,8,13,24,30,34,88,129,305,105,9,736,event,binary
8,target_event_on__earnings_beat,24,17,23,25,16,23,25,25,13,9,191,event,binary
2,target_event_on__insider_buy,2,7,15,17,26,20,26,45,27,8,185,event,binary
4,target_event_on__analyst_upgrade,0,0,0,0,0,0,34,114,66,3,214,event,binary
6,target_event_on__price_target_raise,0,0,0,0,0,0,39,88,75,3,202,event,binary
5,target_event_on__analyst_downgrade,0,0,0,0,0,0,12,22,35,3,69,event,binary
7,target_event_on__price_target_cut,0,0,0,0,0,0,8,22,31,3,61,event,binary


,left,right,intersection,jaccard
31,target_event_on__analyst_upgrade,target_event_on__price_target_raise,139,0.501805
35,target_event_on__analyst_downgrade,target_event_on__price_target_cut,34,0.354167
34,target_event_on__analyst_downgrade,target_event_on__price_target_raise,25,0.101626
32,target_event_on__analyst_upgrade,target_event_on__price_target_cut,24,0.095618
0,target_event_on__congress_buy,target_event_on__congress_sell,225,0.084459
37,target_event_on__price_target_raise,target_event_on__price_target_cut,17,0.069106
30,target_event_on__analyst_upgrade,target_event_on__analyst_downgrade,14,0.052045
24,target_event_on__insider_sell,target_event_on__analyst_upgrade,45,0.049724
2,target_event_on__congress_buy,target_event_on__insider_sell,103,0.049259
26,target_event_on__insider_sell,target_event_on__price_target_raise,37,0.041065


cheap_status,candidate_for_wfo,unstable_annual_support
target_family,,
event,9,1
oracle_trade,1,0


,target,target_family,years_with_support,total_positives,cheap_status
7,target_event_on__price_target_cut,event,3,61,candidate_for_wfo
5,target_event_on__analyst_downgrade,event,3,69,candidate_for_wfo
2,target_event_on__insider_buy,event,8,185,candidate_for_wfo
8,target_event_on__earnings_beat,event,9,191,candidate_for_wfo
6,target_event_on__price_target_raise,event,3,202,candidate_for_wfo
4,target_event_on__analyst_upgrade,event,3,214,candidate_for_wfo
3,target_event_on__insider_sell,event,9,736,candidate_for_wfo
10,target_oracle_trade_entry__long_vs_short,oracle_trade,9,1350,candidate_for_wfo
1,target_event_on__congress_sell,event,9,1431,candidate_for_wfo
0,target_event_on__congress_buy,event,9,1458,candidate_for_wfo


## Gate

Targets with inadequate or unstable label support should not enter the combinatorial model search at larger universes. This is a compute gate, not a claim that sufficiently supported targets are predictively useful. Predictive confirmation belongs in anchored annual WFO in Quant Orchestrator.